# 07 Incumbent-Rebased Delta Judger

06 learned a robust judge, but the original round sequence still drifts after
round 2.  This experiment keeps the best known incumbent and applies only the
late-round update directions onto it:

`I_best + alpha * (A_t - G_t) + beta * (S_t - G_t)`

The judge evaluates body/head/router/expert deltas with the 06 robust proxy and
accepts a full-evaluated candidate only if it improves the incumbent.


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path('/app/Object_Detection')
PROJECT = ROOT / 'dynamic_quality_aware_classwise_aggregation' / 'moe_dqa_judger'
OUT = PROJECT / 'output' / '07_incumbent_delta_judger'
OUT


PosixPath('/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/07_incumbent_delta_judger')

In [2]:
import subprocess, sys

cmd = [
    sys.executable,
    str(PROJECT / 'scripts' / 'run_07_incumbent_delta_judger.py'),
    '--workspace-root', str(OUT),
    '--rounds', '3,4,5,6',
    '--mini-splits', '3',
    '--mini-images', '384',
    '--random-candidates', '4',
    '--full-eval-topk', '2',
    '--val-batch-size', '32',
    '--notify-discord',
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=ROOT, check=True)


/opt/venv/bin/python3 /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_07_incumbent_delta_judger.py --workspace-root /app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/07_incumbent_delta_judger --rounds 3,4,5,6 --mini-splits 3 --mini-images 384 --random-candidates 4 --full-eval-topk 2 --val-batch-size 32 --notify-discord


{
  "manifest": {
    "created_utc": "2026-05-13T14:42:09.186995+00:00",
    "protocol": "dqa_softmox_incumbent_rebased_delta_v1",
    "incumbent": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/03_mix_judger_policy/candidates/r002_judger03_selected_r002.pt",
    "incumbent_score": 0.57455,
    "rounds": [
      3,
      4,
      5,
      6
    ],
    "mini_splits": 3,
    "mini_images": 384,
    "formula": "I_best + alpha*(A_t-G_t) + beta*(S_t-G_t)"
  },
  "accepted": [
    {
      "round": 2,
      "candidate_id": "incumbent_r002",
      "accepted": true,
      "map50": 0.462,
      "map50_95": 0.26,
      "score": 0.57455,
      "path": "/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/03_mix_judger_policy/candidates/r002_judger03_selected_r002.pt",
      "reason": "initial incumbent"
    },
    {
      "round": 3,
      "candidate_id": "rand003",
      "phase": "random",
      "path": "/app/Object_Dete

CompletedProcess(args=['/opt/venv/bin/python3', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/scripts/run_07_incumbent_delta_judger.py', '--workspace-root', '/app/Object_Detection/dynamic_quality_aware_classwise_aggregation/moe_dqa_judger/output/07_incumbent_delta_judger', '--rounds', '3,4,5,6', '--mini-splits', '3', '--mini-images', '384', '--random-candidates', '4', '--full-eval-topk', '2', '--val-batch-size', '32', '--notify-discord'], returncode=0)

In [3]:
summary = pd.read_csv(OUT / 'stats' / '07_summary.csv')
display(summary[['round','candidate_id','phase','mean_score','std_score','proxy_lcb_score','pred_full_score','judger_score']].head(20))

full = pd.read_csv(OUT / 'stats' / '07_full_eval.csv')
display(full[['round','candidate_id','mean_score','pred_full_score','map50','map50_95','score']])

accepted = pd.read_csv(OUT / 'stats' / '07_accepted_policy.csv')
display(accepted[['round','candidate_id','accepted','map50','map50_95','score','incumbent_after_score','reason']])

print((OUT / '07_incumbent_delta_judger_report.md').read_text())


,round,candidate_id,phase,mean_score,std_score,proxy_lcb_score,pred_full_score,judger_score
0,3,tiny_all_s,prior,0.628067,0.041553,0.596902,0.573528,0.573528
1,3,rand003,random,0.628150,0.041627,0.596930,0.573513,0.573513
2,3,scaled04_best01_sur00_00_025,prior,0.628083,0.041700,0.596808,0.573510,0.573510
3,3,scaled04_best00_sur00_02_025,prior,0.628083,0.041700,0.596808,0.573510,0.573510
4,3,target_router_only,prior,0.627933,0.041476,0.596826,0.573505,0.573505
5,3,tiny_all_a,prior,0.627933,0.041476,0.596826,0.573505,0.573505
6,3,moe_a_only,prior,0.627933,0.041476,0.596826,0.573505,0.573505
7,3,head_s_moe_a,prior,0.627933,0.041476,0.596826,0.573505,0.573505
8,3,body_frozen_head_moe,prior,0.627933,0.041476,0.596826,0.573505,0.573505
9,3,source_repair_only_head,prior,0.627933,0.041476,0.596826,0.573505,0.573505


,round,candidate_id,mean_score,pred_full_score,map50,map50_95,score
0,3,tiny_all_s,0.628067,0.573528,0.461,0.26,0.57350
1,3,rand003,0.628150,0.573513,0.462,0.26,0.57405
2,4,scaled04_best00_rand008_050,0.628267,0.572418,0.462,0.26,0.57410
3,4,anti_drift_head,0.628050,0.572412,0.462,0.26,0.57455
4,5,scaled04_best01_sur00_02_025,0.628233,0.572038,0.462,0.26,0.57450
5,5,scaled04_best00_slight_extrapolate_target_025,0.628233,0.572038,0.462,0.26,0.57450
6,6,scaled04_best01_rand000_050,0.628600,0.572057,0.462,0.26,0.57410
7,6,anti_drift_head,0.628033,0.572049,0.462,0.26,0.57455


,round,candidate_id,accepted,map50,map50_95,score,incumbent_after_score,reason
0,2,incumbent_r002,True,0.462,0.26,0.57455,NaN,initial incumbent
1,3,rand003,False,0.462,0.26,0.57405,0.57455,rejected; keep incumbent
2,4,anti_drift_head,False,0.462,0.26,0.57455,0.57455,rejected; keep incumbent
3,5,scaled04_best01_sur00_02_025,False,0.462,0.26,0.57450,0.57455,rejected; keep incumbent
4,6,anti_drift_head,False,0.462,0.26,0.57455,0.57455,rejected; keep incumbent


# DQA-SoftMoX Incumbent-Rebased Delta Judger 07

- created_utc: 2026-05-13T15:10:08.373096+00:00
- incumbent_score: 0.5746
- mini_splits: 3
- mini_images: 384

## Accepted Policy

| round | candidate | accepted | mAP50 | mAP50:95 | score | incumbent_after | reason |
|---:|---|---:|---:|---:|---:|---:|---|
| 2 | incumbent_r002 | True | 0.462 | 0.260 | 0.5746 | 0.5746 | initial incumbent |
| 3 | rand003 | False | 0.462 | 0.260 | 0.5741 | 0.5746 | rejected; keep incumbent |
| 4 | anti_drift_head | False | 0.462 | 0.260 | 0.5746 | 0.5746 | rejected; keep incumbent |
| 5 | scaled04_best01_sur00_02_025 | False | 0.462 | 0.260 | 0.5745 | 0.5746 | rejected; keep incumbent |
| 6 | anti_drift_head | False | 0.462 | 0.260 | 0.5746 | 0.5746 | rejected; keep incumbent |

## Top Proxy Candidates

| rank | round | candidate | proxy LCB | pred full | mean score | std |
|---:|---:|---|---:|---:|---:|---:|
| 1 | 3 | tiny_all_s | 0.5969 | 0.5735 | 0.6281 | 0.0416 |
| 2 | 3 | rand003 | 0.5969 | 0.5735 | 0